In [7]:
# ===========================
# INSTALL (SAFE)
# ===========================
try:
    import gensim.downloader as api
except:
    import os
    os.system('pip install gensim')
    import gensim.downloader as api

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler

import nltk
import re
from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('stopwords')

# ===========================
# STEP 1: DATASET
# ===========================
print("\n" + "="*70)
print("STEP 1: DATASET OVERVIEW")
print("="*70)

tickets = [
    # account
    ("I forgot my password and cannot login", "account"),
    ("My account is locked", "account"),
    ("Unable to reset password", "account"),
    ("Login failed multiple times", "account"),
    ("Need help updating email", "account"),
    ("Authentication not working", "account"),

    # billing
    ("Payment deducted but not activated", "billing"),
    ("Card charged twice", "billing"),
    ("Refund not processed", "billing"),
    ("Invoice missing", "billing"),
    ("Subscription still charged", "billing"),
    ("Payment failed but money deducted", "billing"),

    # technical
    ("App crashes when opened", "technical"),
    ("Error 404 on page", "technical"),
    ("App freezes often", "technical"),
    ("Upload not working", "technical"),
    ("System stuck loading", "technical"),
    ("Feature not working", "technical"),

    # product
    ("What features are in premium", "product"),
    ("How to upgrade plan", "product"),
    ("Tell me pricing", "product"),
    ("Do you offer discounts", "product"),
    ("Explain premium plan", "product"),
    ("Available subscription plans", "product"),

    # shipping
    ("Where is my order", "shipping"),
    ("Product delivered damaged", "shipping"),
    ("Wrong item received", "shipping"),
    ("Order delayed", "shipping"),
    ("Tracking not updated", "shipping"),
    ("Package not delivered", "shipping")
]

texts = [t[0] for t in tickets]
labels = [t[1] for t in tickets]

print(f"Total Samples: {len(texts)}")
print(f"Categories: {set(labels)}")

# ===========================
# STEP 2: PREPROCESSING
# ===========================
print("\n" + "="*70)
print("STEP 2: TEXT PREPROCESSING")
print("="*70)

stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    return [w for w in words if w not in stop_words]

processed = [preprocess(t) for t in texts]

print("Sample:")
print("Original:", texts[0])
print("Processed:", processed[0])

# ===========================
# STEP 3: LOAD EMBEDDINGS
# ===========================
print("\n" + "="*70)
print("STEP 3: LOADING WORD EMBEDDINGS")
print("="*70)

model = api.load('glove-twitter-25')
embedding_dim = model.vector_size

print("Embedding Dimension:", embedding_dim)

# ===========================
# STEP 4: SENTENCE EMBEDDINGS
# ===========================
print("\n" + "="*70)
print("STEP 4: GENERATING EMBEDDINGS")
print("="*70)

def sentence_embedding(words):
    vectors = [model[w] for w in words if w in model]
    if len(vectors) == 0:
        return np.zeros(embedding_dim)
    return np.mean(vectors, axis=0)

X = np.array([sentence_embedding(w) for w in processed])

# scaling improves accuracy
scaler = StandardScaler()
X = scaler.fit_transform(X)

print("Embedding Shape:", X.shape)

# ===========================
# STEP 5: LABEL ENCODING
# ===========================
le = LabelEncoder()
y = le.fit_transform(labels)

print("Label Mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# ===========================
# STEP 6: TRAIN TEST SPLIT (IMPROVED)
# ===========================
print("\n" + "="*70)
print("STEP 6: TRAIN TEST SPLIT")
print("="*70)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42   # CHANGED to 10%
)

print("Train Size:", len(X_train))
print("Test Size:", len(X_test))


# ===========================
# STEP 7: TRAINING (IMPROVED SVM)
# ===========================
print("\n" + "="*70)
print("STEP 7: TRAINING SVM MODEL (TUNED)")
print("="*70)

model_clf = SVC(
    kernel='linear',
    C=10,          # higher margin control
    gamma='scale',
    probability=True
)
print("Training started...")

model_clf.fit(X_train, y_train)

print("Training completed!")


# ===========================
# STEP 8: EVALUATION
# ===========================
print("\n" + "="*70)
print("STEP 8: MODEL EVALUATION")
print("="*70)

pred = model_clf.predict(X_test)
acc = accuracy_score(y_test, pred)

print(f"Accuracy: {acc*100:.2f}%")

from sklearn.metrics import classification_report
print("\nClassification Report:")
print(classification_report(y_test, pred, target_names=le.classes_))

# ===========================
# STEP 9: REAL-TIME TEST
# ===========================
print("\n" + "="*70)
print("STEP 9: REAL-TIME PREDICTION")
print("="*70)

queries = [
    "I cannot login to my account",
    "Payment failed but money deducted",
    "App crashes when opening",
    "Tell me pricing plans",
    "Order not delivered yet"
]

for q in queries:
    p = preprocess(q)
    emb = sentence_embedding(p)
    emb = scaler.transform([emb])

    pred = model_clf.predict(emb)[0]
    category = le.inverse_transform([pred])[0]

    print(f"\nQuery: {q}")
    print("Category:", category.upper())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



STEP 1: DATASET OVERVIEW
Total Samples: 30
Categories: {'account', 'shipping', 'billing', 'product', 'technical'}

STEP 2: TEXT PREPROCESSING
Sample:
Original: I forgot my password and cannot login
Processed: ['forgot', 'password', 'cannot', 'login']

STEP 3: LOADING WORD EMBEDDINGS
Embedding Dimension: 25

STEP 4: GENERATING EMBEDDINGS
Embedding Shape: (30, 25)
Label Mapping: {np.str_('account'): np.int64(0), np.str_('billing'): np.int64(1), np.str_('product'): np.int64(2), np.str_('shipping'): np.int64(3), np.str_('technical'): np.int64(4)}

STEP 6: TRAIN TEST SPLIT
Train Size: 24
Test Size: 6

STEP 7: TRAINING SVM MODEL (TUNED)
Training started...
Training completed!

STEP 8: MODEL EVALUATION
Accuracy: 50.00%

Classification Report:
              precision    recall  f1-score   support

     account       0.50      1.00      0.67         1
     billing       0.33      1.00      0.50         1
     product       0.00      0.00      0.00         1
    shipping       1.00      0.50   